# IP1B: Weather API Practice (OpenWeatherMap)
This notebook builds on Revision Class by focusing entirely on HTTP APIs using a live-ready weather service.

## Summary of Important Concepts
1. **API basics:** APIs bridge apps and servers, similar to a waiter relaying orders.
2. **Protocols and verbs:** Most public APIs ride on HTTP with methods such as GET (read) and POST (write/update).
3. **Response anatomy:** Expect a status code, headers with metadata, and a JSON body containing the payload.
4. **Python tooling:** The requests package sends calls, attaches params/headers, inspects status, and parses JSON.
5. **Weather example:** Supply coordinates or city names plus an API key to fetch real-time climate data.
6. **Error handling:** Use try/except blocks and inspect 4xx vs 5xx codes before trusting the payload.
7. **Rate limits & security:** Watch for 429 errors, keep API keys secret, and authenticate via headers or params.
8. **GET vs POST:** GET retrieves without side effects, while POST sends or mutates data on the server.
9. **Best practices:** Read docs, validate responses, and practice with public endpoints to build confidence.

## Setup Checklist
- Create a free account at https://openweathermap.org/ to obtain an API key (appid).
- Store it as an environment variable named OPENWEATHER_API_KEY or paste it directly in the code when testing locally.
- Install dependencies inside your virtual environment: pip install requests pandas.
- Run each example after updating the key so the HTTP calls return live weather data instead of 401 errors.

In [ ]:
import os

import pandas as pd
import requests

BASE_URL = "https://api.openweathermap.org/data/2.5"
API_KEY = os.environ.get("OPENWEATHER_API_KEY", "YOUR_API_KEY_HERE")

def weather_params(city):
    return {"q": city, "appid": API_KEY, "units": "metric"}

if API_KEY == "YOUR_API_KEY_HERE":
    print("Add your API key first.")
else:
    print("API key found. You can run the next cells.")

This cell keeps setup simple: imports, base URL, and API key in one place.
`weather_params()` avoids repeating query parameters in every request.
If the key is missing, you get a clear reminder before making API calls.
This is enough setup for all examples below.

In [ ]:
city = "Bengaluru"

if API_KEY == "YOUR_API_KEY_HERE":
    print("Add API key and run again.")
else:
    response = requests.get(f"{BASE_URL}/weather", params=weather_params(city), timeout=10)
    if response.status_code == 200:
        data = response.json()
        print("City:", data["name"])
        print("Weather:", data["weather"][0]["main"])
        print("Temperature:", data["main"]["temp"], "C")
        print("Humidity:", data["main"]["humidity"], "%")
    else:
        print("Request failed with status:", response.status_code)

This is the easiest weather call: one city name and one GET request.
A simple status check (`200`) confirms the call worked.
Then we read JSON and print only key fields: weather, temperature, and humidity.
Change the city value to practice quickly.

In [ ]:
params = {
    "lat": 40.7128,
    "lon": -74.0060,
    "appid": API_KEY,
    "units": "metric"
}

if API_KEY == "YOUR_API_KEY_HERE":
    print("Add API key and run again.")
else:
    response = requests.get(f"{BASE_URL}/weather", params=params, timeout=10)
    if response.status_code == 200:
        data = response.json()
        print("Location:", data["name"], data["sys"]["country"])
        print("Wind speed:", data["wind"]["speed"], "m/s")
        print("Cloud cover:", data["clouds"]["all"], "%")
    else:
        print("Request failed with status:", response.status_code)

This example uses latitude and longitude instead of city text.
The response handling is the same: check status, parse JSON, print fields.
It is useful when city names are ambiguous.
Start simple with location, wind, and cloud cover.

In [ ]:
def safe_current_weather(city):
    try:
        response = requests.get(
            f"{BASE_URL}/weather",
            params=weather_params(city),
            timeout=8
        )
        if response.status_code != 200:
            return {"city": city, "error": f"status {response.status_code}"}

        data = response.json()
        return {
            "city": data["name"],
            "status": data["weather"][0]["main"],
            "temp": data["main"]["temp"]
        }
    except Exception as err:
        return {"city": city, "error": str(err)}

print(safe_current_weather("London"))
print(safe_current_weather("InvalidCity123"))

`safe_current_weather()` wraps a request inside `try/except`.
If anything fails, it returns a small error dictionary instead of crashing.
This pattern is beginner-friendly and practical for assignments.
The two sample calls show one success and one failure case.

In [ ]:
def fetch_forecast_df(city, points=5):
    params = weather_params(city)
    params["cnt"] = points

    response = requests.get(f"{BASE_URL}/forecast", params=params, timeout=10)
    if response.status_code != 200:
        return pd.DataFrame([{"error": f"status {response.status_code}"}])

    items = response.json().get("list", [])
    rows = []
    for item in items:
        rows.append({
            "time": item["dt_txt"],
            "temp_C": item["main"]["temp"],
            "condition": item["weather"][0]["main"]
        })

    return pd.DataFrame(rows)

mumbai_forecast = fetch_forecast_df("Mumbai", points=6)
print(mumbai_forecast)

The forecast endpoint returns many entries, so we keep only a few with `cnt`.
Each entry is converted to a small row with time, temperature, and condition.
Finally we return a pandas DataFrame for easy viewing.
This keeps forecasting simple but useful.

In [ ]:
cities = ["Mumbai", "Paris", "San Francisco", "Tokyo"]
results = []

for city in cities:
    response = requests.get(
        f"{BASE_URL}/weather",
        params=weather_params(city),
        timeout=8
    )

    if response.status_code == 200:
        data = response.json()
        results.append({
            "city": data["name"],
            "temp_C": data["main"]["temp"],
            "condition": data["weather"][0]["main"]
        })
    else:
        results.append({"city": city, "error": response.status_code})

batch_df = pd.DataFrame(results)
print(batch_df)

This loop fetches weather for multiple cities one by one.
Each success adds city, temperature, and condition to a list.
Failures still get recorded with status code so nothing is lost.
At the end, the list becomes one clean DataFrame.

## Practice Ideas
- Swap GET for POST using https://httpbin.org/ to rehearse sending JSON payloads and reading mirrored responses.
- Layer in retries with exponential backoff (time.sleep doubled each time) to handle temporary network hiccups.
- Plot the forecast DataFrame with seaborn/matplotlib to visualize trends instead of only printing tables.
- Wire these helpers into a Streamlit or FastAPI mini-app that responds to user-entered cities.